In [1]:
import sys
import os
import time

api_parent_dir = os.path.abspath(
    os.path.join(os.getcwd(), ".."))  # Move up twice
sys.path.append(api_parent_dir)  # Add parent of `api` to sys.path

print("Current sys.path:")
for path in sys.path:
    print(path)

Current sys.path:
/Users/bryantan/.pyenv/versions/3.12.8/lib/python312.zip
/Users/bryantan/.pyenv/versions/3.12.8/lib/python3.12
/Users/bryantan/.pyenv/versions/3.12.8/lib/python3.12/lib-dynload

/Users/bryantan/.pyenv/versions/3.12.8/envs/final_fyp/lib/python3.12/site-packages
/Users/bryantan/Documents/AgentRevamp/agent


## TO BE IN `utils/etf_utils.py`

In [2]:
from constants import CATEGORY_TO_ASSET_CLASS
from typing import Dict, List

import pandas as pd
import numpy as np
from pypfopt import objective_functions
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.efficient_frontier import EfficientFrontier

def classify_etfs(etfs):
    classified_etfs = {}

    for etf, description in etfs.items():
        classified_etfs[etf] = CATEGORY_TO_ASSET_CLASS.get(
            description, 'Others')

    return classified_etfs


def compute_etf_count_by_allocation(allocation: Dict[str, float], total_etfs: int) -> Dict[str, int]:
    """Compute how many ETFs to select per category based on portfolio allocation."""
    return {
        category: max(1, round(total_etfs * weight))
        for category, weight in allocation.items()
    }
    

def overlap_check(selected_etfs, holdings_data):
    overlap_count = {}

    for etf1 in selected_etfs:
        for etf2 in selected_etfs:
            if etf1 != etf2:
                common_holdings = set(holdings_data.get(etf1, [])) & set(
                    holdings_data.get(etf2, []))
                # % of overlapping holdings
                overlap_percentage = (len(common_holdings) / 10) * 100

                overlap_count[(etf1, etf2)] = overlap_percentage

    return overlap_count

### TO BE IN `modules/analyzer/BatchStockAnalyzer`

In [3]:
from collections import defaultdict
from typing import Dict, List

from constants import BOND_CATEGORIES, CATEGORY_TO_ASSET_CLASS
from utils.StockAnalyzerFactory import StockAnalyzerFactory

from yahooquery import Ticker
import yfinance as yf
import pandas as pd

In [4]:
def get_price_data_batch(etfs, period="5y"):
    return yf.download(etfs, period=period)['Close']

def process_etf_batch(portfolio, symbols):
    yq = Ticker(symbols)
    key_stats = yq.key_stats

    etf_categories = {
        symbol: key_stats.get(symbol).get("category", "others")
        for symbol in symbols
    }

    for symbol, category in etf_categories.items():
        print(f"[fetch_category] {symbol}: Category {category}")

    mapped_etfs = classify_etfs(etf_categories)

    filtered_etfs = defaultdict(list)
    for symbol, category in mapped_etfs.items():
        if category in portfolio["allocation"]:
            filtered_etfs[category].append(symbol)

    # Slice top 10 for equity and bond ETFs
    top_10_etfs = {
        cat: etfs[:10] for cat, etfs in filtered_etfs.items() if cat not in BOND_CATEGORIES
    }

    bond_etfs = {
        cat: etfs[:10] for cat, etfs in filtered_etfs.items() if cat in BOND_CATEGORIES
    }

    return top_10_etfs, bond_etfs


def get_category_etf_metrics_batch(all_category_etfs: Dict[str, List[str]]) -> Dict[str, List[dict]]:
    all_tickers = list({ticker for tickers in all_category_etfs.values()
                        for ticker in tickers})

    print(f"[BATCH] Fetching ETF metrics for {all_tickers} tickers")

    etf_metrics = {}
    for ticker in all_tickers:
        try:
            analyzer = StockAnalyzerFactory.get_analyzer(ticker)
            metrics = analyzer.get_etf_metrics()
            if metrics:
                etf_metrics[ticker] = metrics
                print(f"[fetch_metrics] {ticker}: Success")
            else:
                print(f"[fetch_metrics] {ticker}: No data")
        except Exception as e:
            print(f"[ERROR] {ticker} failed: {e}")

    category_metrics: Dict[str, List[dict]] = defaultdict(list)
    for category, tickers in all_category_etfs.items():
        for ticker in tickers:
            if ticker in etf_metrics:
                category_metrics[category].append(etf_metrics[ticker])

    return dict(category_metrics)


def filter_and_sort_etfs(category_metrics: Dict[str, List[dict]]) -> Dict[str, pd.DataFrame]:
    category_dfs: Dict[str, pd.DataFrame] = {}

    for category, metrics in category_metrics.items():
        df = pd.DataFrame(metrics)
        if df.empty:
            continue

        df_filtered = df.dropna(subset=[
                                "Expense_Ratio", "AUM", "Average Volume", "3Y_Return", "5Y_Return", "Sharpe_1Y"])

        if category in BOND_CATEGORIES:
            df_filtered = df_filtered[
                (df_filtered["3Y_Return"] > -0.02) &
                (df_filtered["AUM"] > 500_000_000) &
                (df_filtered["Expense_Ratio"] <= 0.20)
            ]

        elif category == "REITs":
            # 🛠 Relaxed conditions for REITs
            df_filtered = df_filtered[
                (df_filtered["AUM"] > 250_000_000) &
                (df_filtered["Expense_Ratio"] <= 0.30)
            ]

        else:
            df_filtered = df_filtered[
                (df_filtered["3Y_Return"] > 0) &
                (df_filtered["5Y_Return"] > 0) &
                (df_filtered["AUM"] > 500_000_000)
            ]

        df_sorted = df_filtered.sort_values(
            by=["Sharpe_1Y", "AUM", "Average Volume"], ascending=[False, False, False]
        )

        category_dfs[category] = df_sorted

    return category_dfs


def fetch_holdings_batch(tickers: List[str]) -> Dict[str, List[str]]:
    """
    Sequentially fetch top 10 holdings for each ETF in the tickers list.
    Avoids thread pool to reduce API rate limit hits.
    Returns: { ticker: [holdings] }
    """
    holdings_data = {}

    print(f"[BATCH] Fetching holdings for {len(tickers)} ETFs...")

    for ticker in tickers:
        try:
            analyzer = StockAnalyzerFactory.get_analyzer(ticker)
            top_holdings = analyzer.get_etf_holdings_top_10()
            holdings_data[ticker] = top_holdings
            print(f"[fetch_holdings] {ticker}: {len(top_holdings)} holdings")
        except Exception as e:
            print(f"[ERROR] Failed to fetch holdings for {ticker}: {e}")

    return holdings_data


def select_etfs_by_overlap(category: str, df: pd.DataFrame, count: int) -> List[str]:
    """Select ETFs from a category while minimizing holding overlap."""
    tickers = df["Ticker"].tolist(
    )[:count * 2]  # pull extras in case we filter some out

    holdings_data = fetch_holdings_batch(tickers)

    overlap = overlap_check(tickers, holdings_data)
    overlap_df = pd.DataFrame(overlap.items(), columns=[
                              "ETF Pair", "Overlap Percentage"])
    overlap_df = overlap_df.sort_values(
        by="Overlap Percentage", ascending=False)

    overlapping_pairs = overlap_df[overlap_df["Overlap Percentage"]
                                   > 30]["ETF Pair"].tolist()
    filtered = set(tickers)
    for etf1, etf2 in overlapping_pairs:
        if etf1 in filtered and etf2 in filtered:
            filtered.remove(etf2)

    selected = list(filtered)[:count]
    return selected

# TESTING ONLY

In [5]:
from test_data import matched_portfolio, symbols

In [6]:
start_time = time.time()
top_10_etfs, bond_etfs = process_etf_batch(matched_portfolio, symbols)
end_time = time.time()
print(f"Time taken to process ETF batch: {end_time - start_time:.2f} seconds")


[fetch_category] BBUS: Category Large Blend
[fetch_category] SPLG: Category Large Blend
[fetch_category] IVOG: Category Mid-Cap Growth
[fetch_category] SCHR: Category Intermediate Government
[fetch_category] BIV: Category Intermediate Core Bond
[fetch_category] SPTI: Category Intermediate Government
[fetch_category] ILCB: Category Large Blend
[fetch_category] VMBS: Category Intermediate Government
[fetch_category] SPTM: Category Large Blend
[fetch_category] SCHP: Category Inflation-Protected Bond
[fetch_category] VGSH: Category Short Government
[fetch_category] VCSH: Category Short-Term Bond
[fetch_category] STIP: Category Short-Term Inflation-Protected Bond
[fetch_category] IVOV: Category Small Value
[fetch_category] SPTS: Category Short Government
[fetch_category] VCLT: Category Long-Term Bond
[fetch_category] SCHX: Category Large Blend
[fetch_category] ITOT: Category Large Blend
[fetch_category] VGIT: Category Intermediate Government
[fetch_category] IVV: Category Large Blend
[fetch

In [11]:
import json

start_time = time.time()
all_category_etfs = {**top_10_etfs, **bond_etfs}
category_metrics = get_category_etf_metrics_batch(all_category_etfs)
end_time = time.time()

print("Category metrics:")
print(json.dumps(category_metrics, indent=4))
print(f"Time taken for get_category_etf_metrics_batch: {end_time - start_time:.2f} seconds")

[BATCH] Fetching ETF metrics for ['ITOT', 'FREL', 'IVV', 'SPLG', 'VOO', 'VEA', 'SCHX', 'IEFA', 'USRT', 'BBRE', 'VV', 'BBUS', 'PBUS', 'SPEM', 'BIV', 'IDEV', 'ILCB', 'SPTM'] tickers
[fetch_metrics] ITOT: Success
[fetch_metrics] FREL: Success
[fetch_metrics] IVV: Success
[fetch_metrics] SPLG: Success
[fetch_metrics] VOO: Success
[fetch_metrics] VEA: Success
[fetch_metrics] SCHX: Success
[fetch_metrics] IEFA: Success
[fetch_metrics] USRT: Success
[fetch_metrics] BBRE: Success
[fetch_metrics] VV: Success
[fetch_metrics] BBUS: Success
[fetch_metrics] PBUS: Success
[fetch_metrics] SPEM: Success
[fetch_metrics] BIV: Success
[fetch_metrics] IDEV: Success
[fetch_metrics] ILCB: Success
[fetch_metrics] SPTM: Success
Category metrics:
{
    "Large Cap Blend": [
        {
            "Ticker": "BBUS",
            "Name": null,
            "AUM": 4278169856,
            "Average Volume": 232949,
            "Category": "Large Cap Blend",
            "3Y_Return": 0.0856982,
            "5Y_Return": 0.

In [12]:
all_category_etfs

{'Large Cap Blend': ['BBUS',
  'SPLG',
  'ILCB',
  'SPTM',
  'SCHX',
  'ITOT',
  'IVV',
  'PBUS',
  'VOO',
  'VV'],
 'International Large Cap Blend': ['VEA', 'IDEV', 'IEFA'],
 'Emerging Markets': ['SPEM'],
 'REITs': ['USRT', 'FREL', 'BBRE'],
 'Intermediate Bonds': ['BIV']}

In [9]:
start_time = time.time()
category_dfs = filter_and_sort_etfs(category_metrics)
category_counts = compute_etf_count_by_allocation(
    matched_portfolio['allocation'], 10)

final_selected_etfs = []

for category, df in category_dfs.items():
    desired_count = category_counts.get(category, 1)
    selected = select_etfs_by_overlap(category, df, desired_count)
    final_selected_etfs.extend(selected)
    
print("Portfolio before optimizing: ", final_selected_etfs)
end_time = time.time()
print(
    f"Time taken for selecting final etfs: {end_time - start_time:.2f} seconds")

[BATCH] Fetching holdings for 6 ETFs...
[fetch_holdings] SPLG: 10 holdings
[fetch_holdings] VOO: 10 holdings
[fetch_holdings] IVV: 10 holdings
[fetch_holdings] SPTM: 10 holdings
[fetch_holdings] BBUS: 10 holdings
[fetch_holdings] VV: 10 holdings
[BATCH] Fetching holdings for 3 ETFs...
[fetch_holdings] IDEV: 10 holdings
[fetch_holdings] VEA: 10 holdings
[fetch_holdings] IEFA: 10 holdings
[BATCH] Fetching holdings for 1 ETFs...
[fetch_holdings] SPEM: 10 holdings
[BATCH] Fetching holdings for 3 ETFs...
[fetch_holdings] USRT: 10 holdings
[fetch_holdings] BBRE: 10 holdings
[fetch_holdings] FREL: 10 holdings
[BATCH] Fetching holdings for 1 ETFs...
No holdings data available for BIV
[fetch_holdings] BIV: 0 holdings
Portfolio before optimizing:  ['BBUS', 'IDEV', 'SPEM', 'USRT', 'BIV']
Time taken for selecting final etfs: 0.45 seconds


## Optimization PART NEED ABIT OF BRAIN HELP

In [ ]:
def optimize_portfolio(selected_etfs, weights_dict):
    try:
        prices = get_price_data_batch(selected_etfs)
        mu = mean_historical_return(prices)
        S = CovarianceShrinkage(prices).ledoit_wolf()
        ef = EfficientFrontier(mu, S)

        min_w = 0.5 * np.array(list(weights_dict.values()))
        max_w = 1.5 * np.array(list(weights_dict.values()))

        ef.add_constraint(lambda x: x >= min_w)
        ef.add_constraint(lambda x: x <= max_w)
        ef.add_constraint(lambda x: sum(x) == 1)
        ef.add_objective(objective_functions.L2_reg, gamma=0.5)

        ef.efficient_return(mu.mean(), market_neutral=False)
        cleaned = ef.clean_weights()
        return pd.DataFrame(cleaned.items(), columns=["ETF", "Weight"])
    except Exception as e:
        print(f"Optimization error: {e}")
        return pd.DataFrame(weights_dict.items(), columns=["ETF", "Weight"])

In [ ]:
def process_portfolio_optimization(bond_etfs, category_metrics, holdings_data, overlap_df):
    final_equity_etfs = [etf for etf in final_selected_etfs if etf not in bond_etfs]
    final_bond_etfs = [etf for etf in final_selected_etfs if etf in bond_etfs]
    
    etfs_with_weights = {}
    etf_to_category = {}
    
    filtered_equity_df = pd.concat(
        [df for cat, df in category_dfs.items() if cat not in bond_etfs])
    filtered_bond_df = pd.concat(
        [df for cat, df in category_dfs.items() if cat in bond_etfs])
    
    for etf in final_equity_etfs:
        try:
            category = filtered_equity_df[filtered_equity_df["Ticker"] == etf]["Category"].values[0]
            etf_to_category[etf] = category
        except IndexError:
            print(f"[Warning] {etf} not found in equity df")

    for etf in final_bond_etfs:
        try:
            category = filtered_bond_df[filtered_bond_df["Ticker"] == etf]["Category"].values[0]
            etf_to_category[etf] = category
        except IndexError:
            print(f"[Warning] {etf} not found in bond df")
    
    category_counts = {}


    for category in etf_to_category.values():
        category_counts[category] = category_counts.get(category, 0) + 1

    # Assign weights per ETF
    for etf in final_selected_etfs:
        category = etf_to_category.get(etf, "Unknown")
        category_allocation = matched_portfolio["allocation"].get(category, 0)
        count = category_counts.get(category, 1)
        etfs_with_weights[etf] = category_allocation / count
        
        
    try:
        optimized_df = optimize_portfolio(final_selected_etfs, etfs_with_weights, get_price_data)
        portfolio_result = {
        "etfs": final_selected_etfs,
        "weights": optimized_df["Weight"].tolist(),
        "allocation": dict(zip(optimized_df["ETF"], optimized_df["Weight"])),
        "holdings_data": holdings_data,
        "overlap_analysis": overlap_df.to_dict(orient="records")
        }

    except Exception as e:
        print(f"[ERROR] Final portfolio creation failed: {e}")
        portfolio_result = {
            "etfs": final_selected_etfs,
            "weights": list(etfs_with_weights.values()),
            "allocation": etfs_with_weights,
            "holdings_data": holdings_data,
            "overlap_analysis": overlap_df.to_dict(orient="records"),
            "error": str(e)
        }

    print("[RESULT] Final portfolio:")
    print(json.dumps(portfolio_result, indent=2))